# Bytes and Encoding
Foundations for all the crypto topics in this repo. Hash functions, HMAC and digital signatures all work on **bytes**, not text, so this notebook covers how text becomes bytes and how to show bytes to humans.

## Step 1: Text and bytes are different types
`str` is text for humans. `bytes` is what computers store, send and feed into crypto functions. They can look alike when printed, but Python treats them as different things.

In [1]:
text = "hello"
data = b"hello"

print(text, type(text))
print(data, type(data))

# They look alike, but they are not equal
print(text == data)

# Indexing: text gives a character, bytes gives a number (0 to 255)
print(text[0])
print(data[0])
print(b"ABC"[0])
len("é".encode("utf-8"))

hello <class 'str'>
b'hello' <class 'bytes'>
False
h
104
65


2

## Step 2: From characters to numbers to bytes
Every character has a number in Unicode, called a **code point**. An **encoding** such as UTF-8 decides how that number is stored as bytes. Simple English letters take one byte. Many other characters take two, three or four bytes.

In [2]:
# Character to code point, and back
print(ord("h"), chr(104))
print(ord("é"), ord("₦"), ord("😀"))

# Code points to bytes with UTF-8
for ch in ["h", "é", "₦", "😀"]:
    b = ch.encode("utf-8")
    print(f"{ch!r:6} code point {ord(ch):>6}  ->  {len(b)} byte(s): {list(b)}")

104 h
233 8358 128512
'h'    code point    104  ->  1 byte(s): [104]
'é'    code point    233  ->  2 byte(s): [195, 169]
'₦'    code point   8358  ->  3 byte(s): [226, 130, 166]
'😀'    code point 128512  ->  4 byte(s): [240, 159, 152, 128]


In [3]:
# So the number of characters is not always the number of bytes
word = "café"
print(len(word), "characters")
print(len(word.encode("utf-8")), "bytes")

4 characters
5 bytes


## Step 3: Decoding, and what happens with the wrong encoding
`str.encode()` goes from text to bytes. `bytes.decode()` goes back. You must decode with the same encoding you used to encode, or you get wrong text or an error.

In [4]:
data = "café".encode("utf-8")
print(data)
print(data.decode("utf-8")) # correct: back to café

# Wrong encoding: no error, but the text comes out wrong
print(data.decode("latin-1"))

b'caf\xc3\xa9'
café
cafÃ©


In [5]:
# Different encodings give different bytes for the same text
utf8 = "café".encode("utf-8")
latin1 = "café".encode("latin-1")

print(utf8, len(utf8))
print(latin1, len(latin1))

b'caf\xc3\xa9' 5
b'caf\xe9' 4


In [6]:
# Some bytes are simply not valid UTF-8
bad = b"\xff\xfe"

try:
    bad.decode("utf-8")
except UnicodeDecodeError as e:
    print("Error:", e)

Error: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte


### Why this matters for crypto: the same text with different encodings gives a different hash


In [7]:
import hashlib
print(hashlib.sha256("café".encode("utf-8")).hexdigest())
print(hashlib.sha256("café".encode("latin-1")).hexdigest())

850f7dc43910ff890f8879c0ed26fe697c93a067ad93a7d50f466a7028a9bf4e
dafd66c0b98965e688be1fc12942c09f0350e6be0685017c3f234e97d0adc92e


## Step 4: Bytes literals and escape codes
You can write bytes directly with `b"..."`, but only ASCII characters can be typed inside. For any other byte, use an escape code like `\x41` (a byte written as two hex digits).

In [8]:
print(b"hello" == "hello".encode("utf-8"))   # True: ASCII is part of UTF-8

# Escape codes let you write any byte value
print(b"\x68\x65\x6c\x6c\x6f")              # same as b"hello"
print(b"caf\xc3\xa9" == "café".encode("utf-8"))

# Uncomment to see the SyntaxError for non-ASCII inside a bytes literal
# b"café"

True
b'hello'
True


In [9]:
# You can also build bytes from a list of numbers
print(bytes([104, 101, 108, 108, 111]))

# A bytes object is fixed. A bytearray can be changed.
ba = bytearray(b"hello")
ba[0] = ord("j")
print(ba, bytes(ba))

b'hello'
bytearray(b'jello') b'jello'


## Step 5: Showing bytes to humans: hex and Base64
Raw bytes such as hashes, keys and signatures are often unreadable, so we write them as text. **Hex** uses 2 characters per byte. **Base64** is shorter (about 4 characters for every 3 bytes) and is common in emails, JSON and web APIs.

In [10]:
import hashlib, base64

digest = hashlib.sha256(b"hello").digest()
print("Raw bytes:", digest)
print("Length:", len(digest), "bytes")

Raw bytes: b',\xf2M\xba_\xb0\xa3\x0e&\xe8;*\xc5\xb9\xe2\x9e\x1b\x16\x1e\\\x1f\xa7B^s\x043b\x93\x8b\x98$'
Length: 32 bytes


In [11]:
# Hex
hex_str = digest.hex()
print(hex_str, len(hex_str), "characters")
print(bytes.fromhex(hex_str) == digest)          # back to the same bytes

2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824 64 characters
True


In [12]:
# Base64
b64 = base64.b64encode(digest) # NOTE: gives bytes, not str
print(b64, type(b64))
print(b64.decode("ascii"), len(b64), "characters")
print(base64.b64decode(b64) == digest)

b'LPJNul+wow4m6DsqxbninhsWHlwfp0JecwQzYpOLmCQ=' <class 'bytes'>
LPJNul+wow4m6DsqxbninhsWHlwfp0JecwQzYpOLmCQ= 44 characters
True


In [13]:
# Hex, Base64 and raw bytes all hold the same information, just shown differently.
# Useful rule: bytes x 2 = hex characters, bytes x 8 = bits
print(len(digest), "bytes =", len(digest) * 8, "bits =", len(digest.hex()), "hex characters")

32 bytes = 256 bits = 64 hex characters


## Step 6: Common mistakes that cause errors
These are the errors you will meet most often when mixing text and bytes.

In [14]:
import hashlib

# Mistake 1: passing text straight into a hash function

try:
    hashlib.sha256("hello")
except TypeError as e:
    print("Mistake 1:", e)

Mistake 1: Strings must be encoded before hashing


In [15]:
# Mistake 2: joining text and bytes

try:
    "hello " + b"world"
except TypeError as e:
    print("Mistake 2:", e)

# Fix: make both the same type
print("hello " + b"world".decode("utf-8"))
print("hello ".encode("utf-8") + b"world")

Mistake 2: can only concatenate str (not "bytes") to str
hello world
b'hello world'


In [16]:
# Mistake 3: comparing text with bytes always gives False, with no error
digest_hex = hashlib.sha256(b"hello").hexdigest()     # str
digest_raw = hashlib.sha256(b"hello").digest()        # bytes
print(digest_hex == digest_raw)                       # False
print(digest_hex == digest_raw.hex())                 # True

False
True


In [17]:
# Mistake 4: calling str() on bytes does not decode them
data = b"hello"
print(str(data))              # "b'hello'"  (not what you want)
print(data.decode("utf-8"))   # "hello"     (correct)

b'hello'
hello


In [43]:
n = 5

print(format(n, "b"))     # binary, no padding
print(format(n, "08b"))   # binary, 8 characters, zeros in front

print(format(n, "x"))     # hex, no padding
print(format(n, "02x"))   # hex, 2 characters, zeros in front

print(format(n, "8b"))    # 8 characters but no 0, so spaces in front

101
00000101
5
05
     101


In [40]:
n = 255
print("number:", n)
print("bits:  ", format(n, "08b"))   # 8 bits
print("hex:   ", format(n, "02x"))   # 2 hex characters

n = 104   # the letter "h"
print("number:", n)
print("bits:  ", format(n, "08b"))
print("hex:   ", format(n, "02x"))

number: 255
bits:   11111111
hex:    ff
number: 104
bits:   01101000
hex:    68


In [44]:
n = 104
print(f"{n:08b}")   # same as format(n, "08b")
print(f"{n:02x}")   # same as format(n, "02x")
print(f"{n:02X}")   # capital X gives capital hex letters: 68

01101000
68
68


In [45]:
print(bin(5))   # 0b101
print(hex(5))   # 0x5

0b101
0x5


## Summary
- Crypto works on **bytes**. Text must be encoded first, usually with UTF-8.
- Characters become **code points** (numbers), and the encoding turns those into bytes.
- `encode()` goes text to bytes, `decode()` goes bytes to text. Use the same encoding both ways.
- `b"..."` only accepts ASCII; use `\x..` escapes or `.encode("utf-8")` for anything else.
- Hex and Base64 are just readable ways of writing bytes.
- Comparing `str` with `bytes` gives `False` silently, so check your types.